# Lab 8D: Knowledge Engineering in First-Order Logic — Digital Circuits

## Learning goals
By the end of this lab, students can:
- map a circuit domain into an FOL-style ontology: `Gate`, `Type`, `In`, `Out`, `Connected`, and `Signal`;
- run and visualize a full-adder circuit described using structured terms;
- pose ASKVARS-style queries such as "which inputs make sum=0 and carry=1?";
- debug a faulty knowledge base by comparing inferred truth tables.

## Chapter 8 connection
Chapter 8's knowledge-engineering section uses an electronic-circuit domain. It chooses predicates, functions, constants, and axioms, then poses queries to infer output signals for a circuit. This notebook implements that idea visually for a full adder.


In [ ]:
# Run this cell first.

from itertools import product
from html import escape
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not installed. Static circuit examples will be shown instead.")


## 1. FOL-style vocabulary for a full adder

The circuit is represented with terms rather than one flat set of variable names.

Examples:
- `In(1, C1)` means the first input terminal of circuit `C1`.
- `Out(1, X1)` means the first output terminal of gate `X1`.
- `Connected(Out(1,X1), In(1,X2))` states that two terminals share the same signal.
- `Signal(t)` denotes the value, `0` or `1`, at terminal `t`.


In [ ]:
def In(n, c): return ("In", n, c)
def Out(n, c): return ("Out", n, c)

def term_str(t):
    if isinstance(t, tuple) and len(t) == 3:
        return f"{t[0]}({t[1]},{t[2]})"
    return str(t)

GATES_GOOD = {
    "X1": "XOR",
    "X2": "XOR",
    "A1": "AND",
    "A2": "AND",
    "O1": "OR",
}

GATES_BUGGY = dict(GATES_GOOD)
GATES_BUGGY["X2"] = "OR"  # A common ontology/axiom bug: wrong gate type.

CONNECTIONS = [
    (In(1, "C1"), In(1, "X1")),
    (In(2, "C1"), In(2, "X1")),
    (Out(1, "X1"), In(1, "X2")),
    (In(3, "C1"), In(2, "X2")),
    (Out(1, "X2"), Out(1, "C1")),

    (In(1, "C1"), In(1, "A1")),
    (In(2, "C1"), In(2, "A1")),
    (Out(1, "A1"), In(1, "O1")),

    (Out(1, "X1"), In(2, "A2")),
    (In(3, "C1"), In(1, "A2")),
    (Out(1, "A2"), In(2, "O1")),
    (Out(1, "O1"), Out(2, "C1")),
]

# Make connections commutative for propagation.
UNDIRECTED = CONNECTIONS + [(b, a) for a, b in CONNECTIONS]


def gate_output(gtype, inputs):
    if None in inputs:
        return None
    if gtype == "AND": return int(all(inputs))
    if gtype == "OR": return int(any(inputs))
    if gtype == "XOR": return int(inputs[0] != inputs[1])
    if gtype == "NOT": return int(not inputs[0])
    raise ValueError(gtype)


def all_terminals(gates):
    terms = {In(1,"C1"), In(2,"C1"), In(3,"C1"), Out(1,"C1"), Out(2,"C1")}
    for g, typ in gates.items():
        arity = 1 if typ == "NOT" else 2
        for i in range(1, arity + 1):
            terms.add(In(i, g))
        terms.add(Out(1, g))
    for a, b in CONNECTIONS:
        terms.add(a); terms.add(b)
    return terms


def propagate_signals(inputs, gates=GATES_GOOD, max_rounds=20):
    signals = {t: None for t in all_terminals(gates)}
    signals[In(1, "C1")] = inputs[0]
    signals[In(2, "C1")] = inputs[1]
    signals[In(3, "C1")] = inputs[2]
    log = []

    def set_signal(t, value, reason):
        if value is None:
            return False
        if signals.get(t) is None:
            signals[t] = value
            log.append(f"{term_str(t)} = {value}    because {reason}")
            return True
        return False

    for round_no in range(max_rounds):
        changed = False
        # Propagate equality across Connected terms.
        for a, b in UNDIRECTED:
            if signals.get(a) is not None and signals.get(b) is None:
                changed |= set_signal(b, signals[a], f"Connected({term_str(a)}, {term_str(b)})")

        # Apply gate-type axioms.
        for g, typ in gates.items():
            arity = 1 if typ == "NOT" else 2
            ins = [signals.get(In(i, g)) for i in range(1, arity + 1)]
            out = gate_output(typ, ins)
            changed |= set_signal(Out(1, g), out, f"Type({g})={typ} and inputs={ins}")

        if not changed:
            break
    return signals, log


def expected_full_adder(a, b, cin):
    total = a + b + cin
    return total % 2, total // 2

signals, log = propagate_signals((1, 0, 1))
print("Output sum, carry:", signals[Out(1,"C1")], signals[Out(2,"C1")])
print("First five inference steps:")
print("\n".join(log[:5]))


In [ ]:
GATE_POS = {
    "X1": (2.0, 3.2),
    "X2": (4.0, 3.2),
    "A1": (2.0, 1.3),
    "A2": (4.0, 1.3),
    "O1": (6.0, 1.3),
}
TERM_POS = {
    In(1,"C1"): (0.25, 4.0),
    In(2,"C1"): (0.25, 2.8),
    In(3,"C1"): (0.25, 0.65),
    Out(1,"C1"): (7.6, 3.2),
    Out(2,"C1"): (7.6, 1.3),
}

def gate_terminal_pos(t):
    kind, n, c = t
    if c == "C1":
        return TERM_POS[t]
    gx, gy = GATE_POS[c]
    if kind == "In":
        return (gx - 0.55, gy + (0.22 if n == 1 else -0.22))
    return (gx + 0.55, gy)

for t in all_terminals(GATES_GOOD):
    if t not in TERM_POS:
        TERM_POS[t] = gate_terminal_pos(t)


def draw_full_adder(inputs=(1,0,1), gates=GATES_GOOD, title="FOL-described full adder"):
    signals, log = propagate_signals(inputs, gates)
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.axis("off")
    ax.set_title(title)
    ax.set_xlim(-0.2, 8.2)
    ax.set_ylim(0, 4.8)

    # External input/output labels.
    labels = {In(1,"C1"): "A", In(2,"C1"): "B", In(3,"C1"): "Cin", Out(1,"C1"): "SUM", Out(2,"C1"): "CARRY"}
    for term, lab in labels.items():
        x, y = TERM_POS[term]
        ax.text(x, y, f"{lab}\n{signals.get(term)}", ha="center", va="center", bbox=dict(boxstyle="round", fill=False))

    # Gates.
    for g, typ in gates.items():
        x, y = GATE_POS[g]
        rect = plt.Rectangle((x - 0.45, y - 0.35), 0.9, 0.7, fill=False, lw=2)
        ax.add_patch(rect)
        ax.text(x, y, f"{g}\n{typ}\nout={signals.get(Out(1,g))}", ha="center", va="center", fontsize=10)

    # Wires.
    for a, b in CONNECTIONS:
        x1, y1 = TERM_POS[a]
        x2, y2 = TERM_POS[b]
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-", lw=1.6))
        vx = (x1 + x2) / 2
        vy = (y1 + y2) / 2
        value = signals.get(a) if signals.get(a) is not None else signals.get(b)
        ax.text(vx, vy + 0.05, "?" if value is None else str(value), fontsize=9, ha="center")

    sum_bit, carry_bit = signals[Out(1,"C1")], signals[Out(2,"C1")]
    exp_sum, exp_carry = expected_full_adder(*inputs)
    ok = (sum_bit, carry_bit) == (exp_sum, exp_carry)
    ax.text(0.05, 0.1, f"Query result: Signal(Out(1,C1))={sum_bit}, Signal(Out(2,C1))={carry_bit}\nExpected arithmetic result: sum={exp_sum}, carry={exp_carry}\nCircuit correct for this input: {ok}",
            ha="left", va="bottom", fontsize=10)
    plt.show()
    return signals, log

signals, log = draw_full_adder((1, 0, 1))


In [ ]:
def truth_table(gates=GATES_GOOD):
    rows = []
    for inputs in product([0, 1], repeat=3):
        signals, _ = propagate_signals(inputs, gates)
        observed = (signals[Out(1,"C1")], signals[Out(2,"C1")])
        expected = expected_full_adder(*inputs)
        rows.append((*inputs, *observed, *expected, observed == expected))
    return rows


def display_truth_table(gates=GATES_GOOD, title="Truth table"):
    rows = truth_table(gates)
    html = f"<h3>{escape(title)}</h3><table><tr><th>A</th><th>B</th><th>Cin</th><th>inferred SUM</th><th>inferred CARRY</th><th>expected SUM</th><th>expected CARRY</th><th>OK?</th></tr>"
    for row in rows:
        a,b,c,s,carry,es,ec,ok = row
        style = "" if ok else " style='background:#ffd6d6'"
        html += f"<tr{style}><td>{a}</td><td>{b}</td><td>{c}</td><td>{s}</td><td>{carry}</td><td>{es}</td><td>{ec}</td><td>{ok}</td></tr>"
    html += "</table>"
    display(HTML(html))
    return rows

rows = display_truth_table(GATES_GOOD, "Correct full-adder knowledge base")


## 2. ASKVARS-style queries

Chapter 8 asks: which input substitutions make the sum bit `0` and the carry bit `1`? The code below answers the same kind of query by enumerating input substitutions.


In [ ]:
def ask_inputs_for_output(target_sum=0, target_carry=1, gates=GATES_GOOD):
    answers = []
    for inputs in product([0, 1], repeat=3):
        signals, _ = propagate_signals(inputs, gates)
        if (signals[Out(1,"C1")], signals[Out(2,"C1")]) == (target_sum, target_carry):
            answers.append({"i1": inputs[0], "i2": inputs[1], "i3": inputs[2]})
    return answers

answers = ask_inputs_for_output(0, 1)
print("Substitutions for ∃i1,i2,i3 Signal(Out(1,C1))=0 ∧ Signal(Out(2,C1))=1:")
for ans in answers:
    print(ans)


In [ ]:
def interactive_circuit(a=1, b=0, cin=1, target_sum=0, target_carry=1, buggy=False):
    gates = GATES_BUGGY if buggy else GATES_GOOD
    draw_full_adder((a, b, cin), gates, title="Buggy KB: Type(X2)=OR" if buggy else "Correct KB")
    display_truth_table(gates, "Truth table for selected KB")
    answers = ask_inputs_for_output(target_sum, target_carry, gates)
    display(HTML(f"<h3>ASKVARS query</h3><p>Target output: SUM={target_sum}, CARRY={target_carry}</p><pre>{escape(str(answers))}</pre>"))

if WIDGETS_AVAILABLE:
    controls = {
        "a": widgets.Dropdown(options=[0,1], value=1, description="A"),
        "b": widgets.Dropdown(options=[0,1], value=0, description="B"),
        "cin": widgets.Dropdown(options=[0,1], value=1, description="Cin"),
        "target_sum": widgets.Dropdown(options=[0,1], value=0, description="target sum"),
        "target_carry": widgets.Dropdown(options=[0,1], value=1, description="target carry"),
        "buggy": widgets.Checkbox(value=False, description="Use buggy gate type")
    }
    ui = widgets.HBox([
        widgets.VBox([controls["a"], controls["b"], controls["cin"], controls["buggy"]]),
        widgets.VBox([controls["target_sum"], controls["target_carry"]])
    ])
    out = widgets.interactive_output(interactive_circuit, controls)
    display(ui, out)
else:
    interactive_circuit()


## Student practice

1. Turn on the buggy gate type. Which rows fail? Which axiom or fact would you inspect first?
2. Add a `NOT` gate and update `gate_output`. How does the ontology need to change?
3. Write the FOL-style sentence for "connected terminals have equal signals." Then point to where the propagation code implements it.
